# DA3NESTED-GIANT-LARGE-1.1 — Unified Evaluation Pipeline *(Pre-Masked Edition)*

Evaluates **DA3NESTED-GIANT-LARGE-1.1** on the **KITTI Eigen-Split** dataset.

**Key change vs. original:** Binary instance masks are **loaded from the pre-generated
mask store** produced by `inst_seg_notebook.ipynb` — SAM3 is never called at runtime.

### Pipeline
| Step | What happens |
|------|-------------|
| 1 | Index KITTI Eigen-Split (test + optional train) |
| 2 | Load DA3NESTED-GIANT-LARGE-1.1 depth model |
| 3 | For each frame: load masks → run DA3 depth inference → compute scale variants |
| 4 | Write per-mask metrics to CSV |
| 5 | Generate 12 analysis plots |

**Mask store layout** (mirrors KITTI root):
```
MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    <frame_stem>_manifest.json
    <frame_stem>_<super_cat>_<sub_cat>_<idx>_<uid>_<score>_mask.png
```


## 0 · GPU Selection & Imports

In [1]:
# # ── GPU selection ─────────────────────────────────────────────────────────────
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # ← edit as needed

# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import glob
import json
import math
import logging
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision.transforms as T

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# ── Depth Anything 3 ──────────────────────────────────────────────────────────
# pip install -e git+https://github.com/ByteDance-Seed/depth-anything-3
from depth_anything_3.api import DepthAnything3

# NOTE: SAM3 / seg_code imports removed — masks are loaded from disk.
print("Imports OK  |  CUDA available:", torch.cuda.is_available())

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
Imports OK  |  CUDA available: True


## 1 · Configuration  *(edit paths here)*

In [ ]:
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR     = "/STORAGE/Arka/cache"

# ── Dataset ───────────────────────────────────────────────────────────────────
KITTI_ROOT    = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_test"
TEST_ROOT     = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT    = os.path.join(KITTI_ROOT, "train")
USE_TRAIN     = False        # include training split?

# ── Pre-generated mask store ──────────────────────────────────────────────────
# Root produced by inst_seg_notebook.ipynb.
# Directory tree under MASK_ROOT mirrors KITTI_ROOT exactly:
#   MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT     = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output"

# ── Depth clipping ────────────────────────────────────────────────────────────
MIN_DEPTH     = 1e-3
MAX_DEPTH     = 80.0

# ── Camera (KITTI default) ────────────────────────────────────────────────────
FOCAL_LENGTH_X = 721.5577

# ── Mask confidence gate (extra filter on top of SAM3's own threshold) ────────
MIN_MASK_SCORE = 0.50

# ── DA3 model identifier ──────────────────────────────────────────────────────
MODEL_ID      = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"

# ── Output directories ────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output//DA3_nested_giant_1.1_outputs"
OUTPUT_CSV_PATH = os.path.join(MAIN_OUTPUT_DIR, "da3_nested_giant_1.1_results.csv")
PLOT_DIR        = os.path.join(MAIN_OUTPUT_DIR, "plots")
SAVE_NPY        = True      # save raw depth / eval-mask arrays as .npy?

# ── Resume support ────────────────────────────────────────────────────────────
START_INDEX   = 0           # set > 0 to resume from a checkpoint

print("Configuration loaded.")

Configuration loaded.


## 2 · Logging

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da3_eval")
logging.getLogger("transformers").setLevel(logging.ERROR)

## 3 · Metric & Crop Helpers

In [4]:
def compute_depth_errors(gt: np.ndarray,
                         pred: np.ndarray,
                         mask: np.ndarray) -> dict:
    """Standard KITTI / Monodepth2 depth metrics (masked pixels only)."""
    gt_v   = gt[mask]
    pred_v = pred[mask]
    pred_v = np.maximum(pred_v, 1e-8)

    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    return dict(rmse=rmse, abs_rel=abs_rel, sq_rel=sq_rel,
                rmse_log=rmse_log, delta1=delta1, delta2=delta2, delta3=delta3)


def eigen_crop_mask(shape) -> np.ndarray:
    """Eigen-crop region used in standard KITTI depth benchmarking."""
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    top  = 44
    bot  = min(top + 352, h)
    mask[top:bot, 0:w] = True
    return mask


def normalize_depth_for_display(depth: np.ndarray,
                                  cmap=plt.cm.magma) -> np.ndarray:
    """Convert a float depth map to an RGBA uint8 image for visualisation."""
    valid = depth > 0
    if not np.any(valid):
        return np.zeros((*depth.shape, 4), dtype=np.uint8)
    vmin = float(np.percentile(depth[valid], 2))
    vmax = float(np.percentile(depth[valid], 98))
    norm = np.clip((depth - vmin) / max(vmax - vmin, 1e-8), 0.0, 1.0)
    return (cmap(norm) * 255).astype(np.uint8)

print("Metric helpers defined.")

Metric helpers defined.


## 4 · Scale Computation

In [5]:
def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> dict:
    """
    Compute four scale-aligned depth maps from the same raw prediction.

    Returns dict keyed by scale name ('unscaled', 'median', 'mean', 'affine').
    Each value is a dict: scale, shift, depth_map, + all depth-error metrics.
    """
    gt_v    = gt_depth[mask]
    pred_c  = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # ── No Scale ──────────────────────────────────────────────────────────────
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {
        "scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
        "avg_depth": float(np.mean(pred_c[mask])),
        "min_depth": float(np.min(pred_c[mask])),
    }

    # ── Median Scale (ratio of medians) ───────────────────────────────────────
    scale_med = float(np.median(gt_v) / np.median(pred_c[mask]))
    pred_med  = np.clip(pred_c * scale_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {
        "scale": scale_med, "shift": 0.0, "depth_map": pred_med, **m1,
        "avg_depth": float(np.mean(pred_med[mask])),
        "min_depth": float(np.min(pred_med[mask])),
    }

    # ── Mean Scale (ratio of means) ───────────────────────────────────────────
    scale_mean = float(np.mean(gt_v) / np.mean(pred_c[mask]))
    pred_mean  = np.clip(pred_c * scale_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {
        "scale": scale_mean, "shift": 0.0, "depth_map": pred_mean, **m2,
        "avg_depth": float(np.mean(pred_mean[mask])),
        "min_depth": float(np.min(pred_mean[mask])),
    }

    # ── Affine Scale (OLS: scale * pred + shift ≈ GT) ─────────────────────────
    pred_flat        = pred_c[mask].ravel()
    A                = np.vstack([pred_flat, np.ones_like(pred_flat)]).T
    params, _, _, _  = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    scale_aff        = float(params[0])
    shift_aff        = float(params[1])
    pred_aff         = np.clip(scale_aff * pred_c + shift_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {
        "scale": scale_aff, "shift": shift_aff, "depth_map": pred_aff, **m3,
        "avg_depth": float(np.mean(pred_aff[mask])),
        "min_depth": float(np.min(pred_aff[mask])),
    }

    return results

print("Scale computation functions defined.")

Scale computation functions defined.


## 5 · DA3NESTED-GIANT-LARGE-1.1 Model Loading & Inference

In [6]:
_da3_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])


def load_da3_model():
    """Load DA3NESTED-GIANT-LARGE-1.1 and move it to DEVICE."""
    log.info("Loading DA3NESTED-GIANT-LARGE-1.1  (model: %s) …", MODEL_ID)
    mdl = DepthAnything3.from_pretrained(
        MODEL_ID, cache_dir=CACHE_DIR
    ).to(DEVICE).eval()
    log.info("DA3 model ready on %s.", DEVICE)
    return mdl


def run_da3(mdl, im_np: np.ndarray) -> np.ndarray:
    """
    Run DA3NESTED-GIANT-LARGE-1.1 on one RGB image (uint8, H×W×3).
    Resizes to multiples of 14, runs inference, upsamples back.
    Returns float32 metric depth map in metres, shape (H, W).
    """
    orig_h, orig_w = im_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)

    img_pil = Image.fromarray(im_np).resize((new_w, new_h), Image.BICUBIC)
    t_img   = _da3_transform(img_pil)
    inp     = t_img.unsqueeze(0).unsqueeze(1).to(DEVICE)   # (1,1,3,H,W)

    with torch.no_grad():
        preds = mdl(inp, export_feat_layers=[])
        if isinstance(preds, dict):
            depth = preds["depth"]
        elif isinstance(preds, (tuple, list)):
            depth = preds[0]
        else:
            depth = preds

        if isinstance(depth, torch.Tensor):
            if depth.dim() == 3:
                depth = depth.unsqueeze(1)
            depth = F.interpolate(depth, size=(orig_h, orig_w),
                                   mode="bilinear", align_corners=False)

    return depth.squeeze().cpu().numpy().astype(np.float32)

print("DA3 model helpers defined.")

DA3 model helpers defined.


## 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

The two functions below resolve the mask directory for any KITTI frame and
load all pre-generated instance masks from it.  The manifest JSON is used
when available (fast path); otherwise the file names are parsed directly.

In [7]:
def get_mask_dir(rgb_path: str) -> str:
    """
    Derive the pre-generated mask directory for a given RGB image path.

    Mapping:
        KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
           →
        MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /

    Parameters
    ----------
    rgb_path : Absolute path to the KITTI RGB image.

    Returns
    -------
    Absolute path to the frame-level mask directory (may not exist if the
    frame was skipped during mask generation).
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), KITTI_ROOT)
    # e.g. "test/2011_09_26/2011_09_26_drive_0001_sync/image_02/data"
    return os.path.join(MASK_ROOT, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                          super_cat_filter: str = None,
                          sub_cat_filter: str = None) -> list:
    """
    Load all pre-generated instance masks for one frame.

    Prefers the JSON manifest (fast).  Falls back to glob + filename parsing
    when the manifest is absent.

    Parameters
    ----------
    mask_dir         : Absolute path to the frame-level mask directory.
    super_cat_filter : If given, return only instances of this super-category.
    sub_cat_filter   : If given, return only instances of this sub-category.

    Returns
    -------
    List of dicts, each containing:
        uid, local_idx, super_cat, sub_cat, score, box,
        mask (H×W bool ndarray), path (Path object)
    Returns an empty list if the directory does not exist or has no masks.
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    # ── Fast path: use manifest ───────────────────────────────────────────────
    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)

        instances = []
        for rec in records:
            if super_cat_filter and rec["super_cat"] != super_cat_filter:
                continue
            if sub_cat_filter and rec["sub_cat"] != sub_cat_filter:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                log.warning("Mask file missing: %s", mask_file)
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # ── Fallback: glob + filename parsing ─────────────────────────────────────
    # Filename format:
    #   <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mask_file.stem[:-5]          # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]

        parts = suffix.rsplit("_", 3)       # [categories, local_idx, uid, score]
        if len(parts) != 4:
            log.warning("Unexpected mask filename format: %s", mask_file.name)
            continue
        cat_str, idx_str, uid_str, score_str = parts

        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            log.warning("Cannot parse categories from: %s", cat_str)
            continue
        super_cat, sub_cat = cat_parts

        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter and sub_cat != sub_cat_filter:
            continue

        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            log.warning("Cannot parse idx/score from: %s", mask_file.name)
            continue

        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,
            "mask":      mask_np,
            "path":      mask_file,
        })

    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")

Mask loading helpers defined.


## 7 · KITTI Dataset Indexing

In [8]:
def index_kitti(use_train: bool = True) -> pd.DataFrame:
    """
    Walk the KITTI Eigen-Split directory and build a DataFrame of
    (RGB path, GT depth path) pairs for test (and optionally train) frames.
    """
    rows = []

    def _collect(split_root: str, tag: str) -> None:
        pattern  = os.path.join(
            split_root, "*", "*_sync",
            "proj_depth", "groundtruth", "image_02", "*.png"
        )
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts            = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive,
                                    "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(
                folder=tag, sequence=date, drive=drive,
                filename=fname, gt_path=gt_path, rgb_path=rgb_path,
            ))

    _collect(TEST_ROOT, "Test")
    if use_train:
        _collect(TRAIN_ROOT, "Train")

    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.",
             len(df), df["drive"].nunique())
    return df

print("KITTI indexing function defined.")

KITTI indexing function defined.


## 8 · CSV Column Schema

In [9]:
CSV_COLUMNS = [
    # ── Frame & Mask Identity ─────────────────────────────────────────────────
    "img_id",             # integer row index in the dataset DataFrame
    "file_name",          # unique string: folder_drive_frameid
    "mask_id",            # local_idx of this mask within the frame (from manifest)
    "mask_uid",           # globally unique mask UID from SAM3 manifest
    "mask_score",         # SAM3 detection confidence
    "super_cat",          # SAM3 super-category (vehicle / pedestrian / stationary_object)
    "sub_cat",            # SAM3 sub-category (e.g. vehicle_headlights, tree …)
    "horizontal_angle",   # degrees from image centre axis

    # ── Unscaled Depth ────────────────────────────────────────────────────────
    "unscaled_rmse",      "unscaled_abs_rel",   "unscaled_delta1",
    "unscaled_avg_depth", "unscaled_min_depth",

    # ── Median Scale ──────────────────────────────────────────────────────────
    "median_scale",
    "median_rmse",        "median_abs_rel",     "median_delta1",
    "median_avg_depth",   "median_min_depth",

    # ── Mean Scale ────────────────────────────────────────────────────────────
    "mean_scale",
    "mean_rmse",          "mean_abs_rel",       "mean_delta1",
    "mean_avg_depth",     "mean_min_depth",

    # ── Affine Scale ──────────────────────────────────────────────────────────
    "affine_scale",       "affine_shift",
    "affine_rmse",        "affine_abs_rel",     "affine_delta1",
    "affine_avg_depth",   "affine_min_depth",

    # ── LiDAR Ground-Truth Reference ──────────────────────────────────────────
    "lidar_avg_depth",    "lidar_min_depth",

    # ── File Paths ────────────────────────────────────────────────────────────
    "rgb_image_path",     "gt_depth_map_path",
    "da3_pred_map_path",  "mask_path",
]

print(f"CSV_COLUMNS defined ({len(CSV_COLUMNS)} columns).")

CSV_COLUMNS defined (38 columns).


## 9 · Main Processing Loop

In [10]:
def process_dataset(da3_mdl, val_list: pd.DataFrame) -> None:
    """
    Iterate over every frame in val_list:
      1. Load RGB image + LiDAR GT depth.
      2. Load pre-generated SAM3 masks from MASK_ROOT.
      3. Run DA3NESTED-GIANT-LARGE-1.1 depth inference  (only when masks found).
      4. For every mask with sufficient GT coverage, compute all four
         scale variants and append one row to the CSV.
    """
    # ── Create output directories ─────────────────────────────────────────────
    os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    RGB_DIR  = os.path.join(MAIN_OUTPUT_DIR, "rgb_images");   os.makedirs(RGB_DIR,  exist_ok=True)
    GT_DIR   = os.path.join(MAIN_OUTPUT_DIR, "gt_depth");     os.makedirs(GT_DIR,   exist_ok=True)
    DA3_DIR  = os.path.join(MAIN_OUTPUT_DIR, "da3_pred");     os.makedirs(DA3_DIR,  exist_ok=True)
    MASK_OUT = os.path.join(MAIN_OUTPUT_DIR, "masks");        os.makedirs(MASK_OUT, exist_ok=True)

    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)

    frames_skipped_no_masks = 0

    for index, row in tqdm(val_list.iterrows(), total=len(val_list),
                           desc="DA3 Inference + Pre-Mask Loading"):
        if index < START_INDEX:
            continue

        base_fn = (os.path.splitext(row["folder"])[0]
                   + "_" + os.path.splitext(row["drive"])[0]
                   + "_" + os.path.splitext(row["filename"])[0])

        try:
            # ── Load image & GT depth ─────────────────────────────────────────
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = (np.array(Image.open(row["gt_path"]))
                        .astype(np.uint16) / 256.0)   # 16-bit KITTI → metres

            H, W, _ = image_np.shape
            cx       = W / 2.0

            # ── Load pre-generated masks from disk ────────────────────────────
            mask_dir      = get_mask_dir(row["rgb_path"])
            all_instances = load_masks_for_frame(mask_dir)

            if not all_instances:
                frames_skipped_no_masks += 1
                log.debug("No pre-generated masks for %s (dir: %s)",
                          base_fn, mask_dir)
                continue

            # ── DA3 depth inference (only if masks exist) ─────────────────────
            pred_raw = run_da3(da3_mdl, image_np)

            # ── Save RGB and depth maps (once per frame) ──────────────────────
            rgb_path_out = os.path.join(RGB_DIR,  f"{base_fn}.png")
            gt_path_out  = os.path.join(GT_DIR,   f"{base_fn}.npy")
            da3_path_out = os.path.join(DA3_DIR,  f"{base_fn}.npy")
            Image.fromarray(image_np).save(rgb_path_out)
            if SAVE_NPY:
                np.save(gt_path_out,  gt_depth)
                np.save(da3_path_out, pred_raw)

            crop_mask = eigen_crop_mask(gt_depth.shape)

            # ── Per-mask computation ──────────────────────────────────────────
            for inst in all_instances:
                seg_mask  = inst["mask"]
                score     = inst["score"]
                super_cat = inst["super_cat"]
                sub_cat   = inst["sub_cat"]
                mask_idx  = inst["local_idx"]
                mask_uid  = inst.get("uid", "")

                if score < MIN_MASK_SCORE:
                    continue

                # Validate spatial dimensions
                if seg_mask.shape != (H, W):
                    log.warning(
                        "Mask shape %s != image shape %s for %s (uid=%s); skipping.",
                        seg_mask.shape, (H, W), base_fn, mask_uid,
                    )
                    continue

                # Evaluation mask: object pixels ∩ valid GT ∩ Eigen crop
                final_mask = (seg_mask
                              & (gt_depth > MIN_DEPTH)
                              & (gt_depth < MAX_DEPTH)
                              & crop_mask)
                if not np.any(final_mask):
                    continue

                scale_res  = compute_all_scales(gt_depth, pred_raw, final_mask)

                lidar_vals = gt_depth[final_mask]
                lidar_avg  = float(np.mean(lidar_vals))
                lidar_min  = float(np.min(lidar_vals))

                _, x_coords = np.where(seg_mask)
                centroid_x  = float(np.mean(x_coords)) if len(x_coords) > 0 else cx
                angle_deg   = math.degrees(
                    math.atan((centroid_x - cx) / FOCAL_LENGTH_X)
                )

                mask_path_out = str(inst.get("path", ""))
                if SAVE_NPY and mask_path_out:
                    npy_mask_out = os.path.join(
                        MASK_OUT, f"{base_fn}_mask_{mask_idx}.npy"
                    )
                    np.save(npy_mask_out, final_mask)

                r = scale_res
                csv_row = {
                    "img_id":             index,
                    "file_name":          base_fn,
                    "mask_id":            mask_idx,
                    "mask_uid":           mask_uid,
                    "mask_score":         score,
                    "super_cat":          super_cat,
                    "sub_cat":            sub_cat,
                    "horizontal_angle":   angle_deg,
                    "unscaled_rmse":      r["unscaled"]["rmse"],
                    "unscaled_abs_rel":   r["unscaled"]["abs_rel"],
                    "unscaled_delta1":    r["unscaled"]["delta1"],
                    "unscaled_avg_depth": r["unscaled"]["avg_depth"],
                    "unscaled_min_depth": r["unscaled"]["min_depth"],
                    "median_scale":       r["median"]["scale"],
                    "median_rmse":        r["median"]["rmse"],
                    "median_abs_rel":     r["median"]["abs_rel"],
                    "median_delta1":      r["median"]["delta1"],
                    "median_avg_depth":   r["median"]["avg_depth"],
                    "median_min_depth":   r["median"]["min_depth"],
                    "mean_scale":         r["mean"]["scale"],
                    "mean_rmse":          r["mean"]["rmse"],
                    "mean_abs_rel":       r["mean"]["abs_rel"],
                    "mean_delta1":        r["mean"]["delta1"],
                    "mean_avg_depth":     r["mean"]["avg_depth"],
                    "mean_min_depth":     r["mean"]["min_depth"],
                    "affine_scale":       r["affine"]["scale"],
                    "affine_shift":       r["affine"]["shift"],
                    "affine_rmse":        r["affine"]["rmse"],
                    "affine_abs_rel":     r["affine"]["abs_rel"],
                    "affine_delta1":      r["affine"]["delta1"],
                    "affine_avg_depth":   r["affine"]["avg_depth"],
                    "affine_min_depth":   r["affine"]["min_depth"],
                    "lidar_avg_depth":    lidar_avg,
                    "lidar_min_depth":    lidar_min,
                    "rgb_image_path":     rgb_path_out,
                    "gt_depth_map_path":  gt_path_out,
                    "da3_pred_map_path":  da3_path_out,
                    "mask_path":          mask_path_out,
                }
                pd.DataFrame([csv_row]).to_csv(
                    OUTPUT_CSV_PATH, mode="a", header=False, index=False
                )

        except Exception as exc:
            log.error("Error processing %s: %s", base_fn, exc, exc_info=True)

    log.info("Processing complete → %s", OUTPUT_CSV_PATH)
    if frames_skipped_no_masks > 0:
        log.warning(
            "%d frames skipped (no pre-generated masks in MASK_ROOT).",
            frames_skipped_no_masks,
        )

print("process_dataset() defined.")

process_dataset() defined.


## 10 · Analysis & Visualisation Helpers

In [11]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale",
    "mean":     "Mean Scale",
    "affine":   "Affine Scale",
}


def load_results() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_CSV_PATH)
    df = df.dropna(subset=[
        "unscaled_rmse", "median_rmse", "mean_rmse", "affine_rmse",
        "lidar_avg_depth", "lidar_min_depth",
    ])
    return df


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)

print("Analysis helpers defined.")

Analysis helpers defined.


## 11 · Plot Functions

In [12]:
# ── Plot 01: GT vs Predicted Depth ────────────────────────────────────────────
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [
        ("Unscaled",     "lidar_avg_depth", "unscaled_avg_depth"),
        ("Median Scale", "lidar_avg_depth", "median_avg_depth"),
        ("Mean Scale",   "lidar_avg_depth", "mean_avg_depth"),
        ("Affine Scale", "lidar_avg_depth", "affine_avg_depth"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    for ax, (title, xcol, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x=xcol, y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_title(f"DA3NESTED-GIANT-1.1  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("Ground-Truth vs Predicted Depth — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DA3NESTED-GIANT-1.1  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    fig.suptitle("GT Depth vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DA3NESTED-GIANT-1.1  |  {title}: Angle vs RMSE", fontsize=12)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)"); ax.set_ylim(0, ymax)
    fig.suptitle("Horizontal Angle vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bar Chart ────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  ↓",["unscaled_rmse","median_rmse","mean_rmse","affine_rmse"]),
        ("Abs Rel   ↓",["unscaled_abs_rel","median_abs_rel","mean_abs_rel","affine_abs_rel"]),
        ("δ < 1.25  ↑",["unscaled_delta1","median_delta1","mean_delta1","affine_delta1"]),
    ]
    labels = ["Unscaled","Median","Mean","Affine"]
    colors = list(SCALE_COLORS.values())
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
        ax.set_title(metric_name, fontsize=13); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.22); ax.grid(axis="y", alpha=0.35)
    fig.suptitle("DA3NESTED-GIANT-1.1 — Scale Method Comparison", fontsize=15, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Model vs LiDAR by Object Category ───────────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    n = len(cats)
    if n == 0: return
    fig, axes = plt.subplots(1, max(n, 1), figsize=(7 * n, 6))
    if n == 1: axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ["unscaled","median","mean","affine"]:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=8)
    fig.suptitle("Predicted Depth vs LiDAR GT — by Object Category", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10)); axes = axes.flatten()
    for ax, (label, col), color in zip(axes, variants, SCALE_COLORS.values()):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(), color="black", linestyle="--", linewidth=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey", linestyle=":", linewidth=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {label}", fontsize=12)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("RMSE Distributions Across All Masks", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", linestyle=":", linewidth=1.8,
                   label=f"Mean = {vals.mean():.3f}")
        ax.set_title(f"{label} Distribution", fontsize=12)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor Distributions", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    variants = [("Unscaled","unscaled_rmse","b"),("Median Scale","median_rmse","g"),
                ("Mean Scale","mean_rmse","orange"),("Affine Scale","affine_rmse","purple")]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [v[1] for v in variants])
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_min = df_c[[v[1] for v in variants]].min().min()
    z_max = df_c[[v[1] for v in variants]].max().max()
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(26, 6))
    fig.suptitle(f"DA3NESTED-GIANT-1.1 — 3D Error Analysis (poly deg={degree})", fontsize=14)
    for sp_idx, (label, zcol, color) in enumerate(variants, start=1):
        ax = fig.add_subplot(1, 4, sp_idx, projection="3d")
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=11)
        bx, by, bz = buf(x_max-x_min), buf(y_max-y_min), buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by)
        ax.set_zlim(z_min-bz, z_max+bz)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ["unscaled","median","mean","affine"]:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin — Scale Comparison")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ["unscaled","median","mean","affine"]:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin (Pred − LiDAR)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("Depth-Bin Analysis — DA3NESTED-GIANT-1.1", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 2: return
    keys, n_keys = ["unscaled","median","mean","affine"], 4
    x, w = np.arange(len(cats)), 0.18
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(keys):
        vals   = [df[df["super_cat"] == c][f"{key}_rmse"].mean() for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE per Object Category — DA3NESTED-GIANT-1.1", fontsize=13)
    ax.legend(); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs Depth ───────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Scale = 1.0")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor vs Ground-Truth Depth", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df: pd.DataFrame) -> None:
    rows_data, specs = [], [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift"),
    ]
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        rows_data.append([label, f"{sv:.4f}", f"{shv:.4f}" if shc else "—",
                          f"{df[rc].mean():.4f}", f"{df[ac].mean():.4f}",
                          f"{df[dc].mean():.4f}"])
    fig, ax = plt.subplots(figsize=(14, 3)); ax.axis("off")
    tbl = ax.table(cellText=rows_data,
                   colLabels=["Scale Method","Avg Scale","Avg Shift",
                               "Avg RMSE (m)","Avg Abs Rel","Avg δ<1.25"],
                   cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 1.9)
    for j in range(6):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows_data) + 1):
        bg = "white" if i % 2 == 0 else "#f0f4f8"
        for j in range(6):
            tbl[i, j].set_facecolor(bg)
    fig.suptitle("DA3NESTED-GIANT-LARGE-1.1 — Scale Method Summary",
                 fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


def generate_all_plots(df: pd.DataFrame) -> None:
    """Generate and save all 12 analysis plots."""
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots → %s", PLOT_DIR)
    plot_summary_table(df);             plot_gt_vs_pred(df)
    plot_gt_vs_rmse(df);                plot_angle_vs_rmse(df)
    plot_scale_comparison_bars(df);     plot_model_vs_lidar_by_category(df)
    plot_rmse_distributions(df);        plot_scale_distributions(df)
    plot_3d_error_surfaces(df);         plot_depth_bin_analysis(df)
    plot_rmse_by_category(df);          plot_scale_vs_depth(df)
    log.info("All 12 plots saved to %s", PLOT_DIR)

print("All plot functions defined.")

All plot functions defined.


## 12 · Terminal Summary Printer

In [13]:
def print_summary(df: pd.DataFrame) -> None:
    W = 105
    print("\n" + "═" * W)
    print("  DA3NESTED-GIANT-LARGE-1.1 — Evaluation Summary")
    print(f"  Masks evaluated: {len(df):,}   |   Drives: {df['img_id'].nunique():,}")
    print("═" * W)
    hdr = (f"{'Scale Method':<24}"
           f"{'Avg Scale':>12}{'Avg Shift':>12}"
           f"{'RMSE (m)':>12}{'Abs Rel':>12}"
           f"{'δ<1.25':>10}{'LiDAR avg':>12}{'LiDAR min':>12}")
    print(hdr); print("─" * W)
    specs = [
        ("Unscaled",     "unscaled_rmse","unscaled_abs_rel","unscaled_delta1",None,None),
        ("Median Scale", "median_rmse",  "median_abs_rel",  "median_delta1",  "median_scale",None),
        ("Mean Scale",   "mean_rmse",    "mean_abs_rel",    "mean_delta1",    "mean_scale",  None),
        ("Affine Scale", "affine_rmse",  "affine_abs_rel",  "affine_delta1",  "affine_scale","affine_shift"),
    ]
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        print(
            f"  {label:<22}"
            f"{sv:>12.4f}{shv:>12.4f}"
            f"{df[rc].mean():>12.4f}{df[ac].mean():>12.4f}"
            f"{df[dc].mean():>10.4f}"
            f"{df['lidar_avg_depth'].mean():>12.4f}"
            f"{df['lidar_min_depth'].mean():>12.4f}"
        )
    print("═" * W + "\n")

print("print_summary() defined.")

print_summary() defined.


## 13 · Run Inference  *(skip if CSV already exists)*

In [14]:
# ── Set SKIP_INFERENCE = True to go straight to analysis from an existing CSV ─
SKIP_INFERENCE = False

if not SKIP_INFERENCE:
    log.info("── STEP 1/3: Indexing KITTI Eigen Split dataset …")
    val_list = index_kitti(use_train=USE_TRAIN)

    log.info("── STEP 2/3: Loading DA3NESTED-GIANT-LARGE-1.1 model …")
    da3_mdl = load_da3_model()
    # No SAM3 model to load — masks are read directly from MASK_ROOT

    log.info("── STEP 3/3: Processing dataset (masks from %s) …", MASK_ROOT)
    process_dataset(da3_mdl, val_list)
else:
    log.info("Skipping inference — will use existing CSV at %s", OUTPUT_CSV_PATH)

15:29:35  INFO      ── STEP 1/3: Indexing KITTI Eigen Split dataset …


Indexing Test:   0%|          | 0/652 [00:00<?, ?it/s]

15:29:35  INFO      Dataset indexed: 652 frames, 28 drives.
15:29:35  INFO      ── STEP 2/3: Loading DA3NESTED-GIANT-LARGE-1.1 model …
15:29:35  INFO      Loading DA3NESTED-GIANT-LARGE-1.1  (model: depth-anything/DA3NESTED-GIANT-LARGE-1.1) …
15:29:35  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3NESTED-GIANT-LARGE-1.1/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
15:29:35  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3NESTED-GIANT-LARGE-1.1/b2359bdf726fb44ef62acca04d629dcf158053e7/config.json "HTTP/1.1 200 OK"


[INFO ] using SwiGLU layer as FFN
[INFO ] using MLP layer as FFN


15:29:50  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3NESTED-GIANT-LARGE-1.1/resolve/main/model.safetensors "HTTP/1.1 302 Found"
15:30:05  INFO      DA3 model ready on cuda.
15:30:05  INFO      ── STEP 3/3: Processing dataset (masks from /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output) …


DA3 Inference + Pre-Mask Loading:   0%|          | 0/652 [00:00<?, ?it/s]

19:42:00  INFO      Processing complete → /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output//DA3_nested_giant_1.1_outputs/da3_nested_giant_1.1_results.csv
19:42:00  WARNING   4 frames skipped (no pre-generated masks in MASK_ROOT).


## 14 · Analysis & Plots

In [15]:
# ── Set SKIP_PLOTS = True to only print summary without regenerating plots ─────
SKIP_PLOTS = False

if os.path.exists(OUTPUT_CSV_PATH):
    log.info("Loading results CSV …")
    df = load_results()
    print_summary(df)

    if not SKIP_PLOTS:
        generate_all_plots(df)
else:
    log.warning(
        "Results CSV not found at %s.  "
        "Run the inference cell first.",
        OUTPUT_CSV_PATH,
    )

19:42:00  INFO      Loading results CSV …
19:42:00  INFO      Generating analysis plots → /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output//DA3_nested_giant_1.1_outputs/plots



═════════════════════════════════════════════════════════════════════════════════════════════════════════
  DA3NESTED-GIANT-LARGE-1.1 — Evaluation Summary
  Masks evaluated: 8,221   |   Drives: 645
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method               Avg Scale   Avg Shift    RMSE (m)     Abs Rel    δ<1.25   LiDAR avg   LiDAR min
─────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                    1.0000      0.0000      8.1470      0.2425    0.5141     31.3927     26.6165
  Median Scale                1.2490      0.0000      2.9886      0.0551    0.9499     31.3927     26.6165
  Mean Scale                  1.2352      0.0000      2.7388      0.0592    0.9505     31.3927     26.6165
  Affine Scale                0.7829     15.3208      1.6441      0.0348    0.9789     31.3927     26.6165
══════════════════════════════════════════════════════

19:42:01  INFO        ✓  00_summary_table.png
19:42:21  INFO        ✓  01_gt_vs_pred_all_scales.png
19:42:38  INFO        ✓  02_gt_vs_rmse_all_scales.png
19:42:58  INFO        ✓  03_angle_vs_rmse_all_scales.png
19:42:59  INFO        ✓  04_scale_comparison_bars.png
19:43:00  INFO        ✓  05_model_vs_lidar_by_category.png
19:43:02  INFO        ✓  06_rmse_distributions.png
19:43:03  INFO        ✓  07_scale_distributions.png
19:43:07  INFO        ✓  08_3d_error_surfaces.png
19:43:09  INFO        ✓  09_depth_bin_analysis.png
19:43:09  INFO        ✓  10_rmse_by_category.png
19:43:24  INFO        ✓  11_scale_vs_depth.png
19:43:24  INFO      All 12 plots saved to /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output//DA3_nested_giant_1.1_outputs/plots
